In [2]:
"""Fetch and parse current matches from Cricbuzz's live-scores page."""

from __future__ import annotations

import json
import re
from typing import Any

import requests


LIVE_SCORES_URL = "https://www.cricbuzz.com/cricket-match/live-scores"
DEFAULT_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
}


def _next_payload_chunks(page: str) -> list[str]:
    """Decode the string chunks emitted by Next.js into the HTML."""
    chunks = []
    pattern = re.compile(r"self\.__next_f\.push\((\[1,\s*\"(?:\\.|[^\"\\])*\"\])\)")
    for match in pattern.finditer(page):
        try:
            value = json.loads(match.group(1))
        except json.JSONDecodeError:
            continue
        if len(value) == 2 and isinstance(value[1], str):
            chunks.append(value[1])
    return chunks


def _extract_current_matches(page: str) -> list[dict[str, Any]]:
    marker = '"currentMatchesList":'

    for chunk in _next_payload_chunks(page):
        marker_index = chunk.find(marker)
        if marker_index == -1:
            continue

        value_start = marker_index + len(marker)
        try:
            current_matches, _ = json.JSONDecoder().raw_decode(chunk[value_start:])
        except json.JSONDecodeError:
            continue

        matches = []
        for type_group in current_matches.get("typeMatches", []):
            match_type = type_group.get("matchType")
            for series_group in type_group.get("seriesMatches", []):
                series = series_group.get("seriesAdWrapper") or {}
                for raw_match in series.get("matches", []):
                    match_info = raw_match.get("matchInfo") or {}
                    if not match_info:
                        continue
                    matches.append(
                        _normalise_match(
                            match_info,
                            raw_match.get("matchScore") or {},
                            match_type,
                            series,
                        )
                    )
        return matches

    raise ValueError("Cricbuzz match data was not present in the page")


def _normalise_match(
    info: dict[str, Any],
    score: dict[str, Any],
    match_type: str | None,
    series: dict[str, Any],
) -> dict[str, Any]:
    match_id = info.get("matchId")
    venue = info.get("venueInfo") or {}
    return {
        "match_id": match_id,
        "series_id": info.get("seriesId", series.get("seriesId")),
        "series_name": info.get("seriesName", series.get("seriesName")),
        "match_type": match_type,
        "description": info.get("matchDesc"),
        "format": info.get("matchFormat"),
        "state": info.get("state"),
        "status": info.get("status"),
        "start_time_ms": _as_int(info.get("startDate")),
        "end_time_ms": _as_int(info.get("endDate")),
        "team1": _normalise_team(info.get("team1"), score.get("team1Score")),
        "team2": _normalise_team(info.get("team2"), score.get("team2Score")),
        "venue": {
            "ground": venue.get("ground"),
            "city": venue.get("city"),
            "timezone": venue.get("timezone"),
        },
        "url": f"https://www.cricbuzz.com/live-cricket-scores/{match_id}" if match_id else None,
    }


def _normalise_team(team: Any, team_score: Any) -> dict[str, Any]:
    team = team if isinstance(team, dict) else {}
    team_score = team_score if isinstance(team_score, dict) else {}
    innings = []
    for innings_key, value in team_score.items():
        if not isinstance(value, dict):
            continue
        innings.append(
            {
                "innings": innings_key,
                "innings_id": value.get("inningsId"),
                "runs": value.get("runs"),
                "wickets": value.get("wickets"),
                "overs": value.get("overs"),
            }
        )
    return {
        "team_id": team.get("teamId"),
        "name": team.get("teamName"),
        "short_name": team.get("teamSName"),
        "innings": innings,
    }


def _as_int(value: Any) -> int | None:
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def get_live_matches(
    match_id: int | str | None = None,
    *,
    timeout: float = 15,
    session: requests.Session | None = None,
) -> dict[str, Any]:
    """Return current Cricbuzz matches in a JSON-serializable dictionary.

    If ``match_id`` is supplied, the result contains only that match. A missing
    ID is reported with ``status='not_found'`` rather than raising an exception.
    """
    client = session or requests
    try:
        response = client.get(
            LIVE_SCORES_URL,
            headers=DEFAULT_HEADERS,
            timeout=timeout,
        )
        response.raise_for_status()
        matches = _extract_current_matches(response.text)
    except (requests.RequestException, ValueError) as exc:
        return {
            "status": "error",
            "message": "Unable to retrieve live matches from Cricbuzz",
            "matches": [],
            "error": str(exc),
        }

    if match_id is not None:
        wanted_id = str(match_id)
        matches = [match for match in matches if str(match.get("match_id")) == wanted_id]
        if not matches:
            return {
                "status": "not_found",
                "message": f"Match {match_id} was not found",
                "matches": [],
            }

    if not matches:
        return {
            "status": "no_matches",
            "message": "There are no current matches",
            "matches": [],
        }

    return {
        "status": "ok",
        "count": len(matches),
        "matches": matches,
    }


def get_live_matches_json(match_id: int | str | None = None, **kwargs: Any) -> str:
    """Return :func:`get_live_matches` as a JSON string."""
    return json.dumps(get_live_matches(match_id, **kwargs), ensure_ascii=False)


get_live_matches()

{'status': 'ok',
 'count': 39,
 'matches': [{'match_id': 129585,
   'series_id': 10565,
   'series_name': 'Pakistan tour of England 2026',
   'match_type': 'International',
   'description': '2nd Test',
   'format': 'TEST',
   'state': 'Stumps',
   'status': 'Day 3: Stumps - England lead by 357 runs',
   'start_time_ms': 1787824800000,
   'end_time_ms': 1788195600000,
   'team1': {'team_id': 9,
    'name': 'England',
    'short_name': 'ENG',
    'innings': [{'innings': 'inngs1',
      'innings_id': 1,
      'runs': 290,
      'wickets': 10,
      'overs': 66.4},
     {'innings': 'inngs2',
      'innings_id': 3,
      'runs': 177,
      'wickets': 7,
      'overs': 46.5}]},
   'team2': {'team_id': 3,
    'name': 'Pakistan',
    'short_name': 'PAK',
    'innings': [{'innings': 'inngs1',
      'innings_id': 2,
      'runs': 110,
      'wickets': 10,
      'overs': 37.2}]},
   'venue': {'ground': "Lord's", 'city': 'London', 'timezone': '+01:00'},
   'url': 'https://www.cricbuzz.com/live-cr

In [3]:
match_id = 129585

In [8]:
def get_live_score(match_id):
    if not match_id:
        raise ValueError("match_id is required")

    scorecard_url = f'https://www.cricbuzz.com/api/mcenter/scorecard/{match_id}'

    response = requests.get(scorecard_url, headers=DEFAULT_HEADERS)
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch scorecard for match_id {match_id}: {response.status_code}")

    return response.json()

get_live_score(match_id)

{'scoreCard': [{'matchId': 129585,
   'inningsId': 1,
   'timeScore': 1787915271488,
   'batTeamDetails': {'batTeamId': 9,
    'batTeamName': 'England',
    'batTeamShortName': 'ENG',
    'batsmenData': {'bat_1': {'batId': 8502,
      'batName': 'Ben Duckett',
      'batShortName': 'Ben Duckett',
      'isCaptain': False,
      'isKeeper': False,
      'runs': 17,
      'balls': 20,
      'dots': 14,
      'fours': 3,
      'sixes': 0,
      'mins': 32,
      'strikeRate': 85,
      'outDesc': 'lbw b Mohammad Abbas',
      'bowlerId': 11298,
      'fielderId1': 0,
      'fielderId2': 0,
      'fielderId3': 0,
      'ones': 1,
      'twos': 2,
      'threes': 0,
      'fives': 0,
      'boundaries': 3,
      'sixers': 0,
      'wicketCode': 'LBW',
      'isOverseas': False,
      'inMatchChange': '',
      'playingXIChange': ''},
     'bat_2': {'batId': 14819,
      'batName': 'Emilio Gay',
      'batShortName': 'Emilio Gay',
      'isCaptain': False,
      'isKeeper': False,
      'run

In [13]:
import requests
import json
import time


def get_all_overs(match_id, innings_id, headers=None, delay=0.5):
    """
    Fetch over-by-over data from Cricbuzz, following pagination until complete.

    Args:
        match_id (int/str): Cricbuzz match ID
        innings_id (int/str): Innings number
        headers (dict): Request headers (uses a default set if None)
        delay (float): Seconds to wait between requests

    Returns:
        dict: Combined JSON with all overs merged into one list
    """
    if headers is None:
        headers = {
            'accept': '*/*',
            'accept-language': 'en-US,en;q=0.9',
            'cache-control': 'no-cache',
            'content-type': 'application/json',
            'pragma': 'no-cache',
            'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                           '(KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36',
        }

    base_url = "https://www.cricbuzz.com"
    url = f"{base_url}/api/mcenter/over-by-over/{match_id}/{innings_id}"

    all_overs = []
    last_response = None
    overs_key = "paginatedData"

    while url:
        print(f"Fetching: {url}")
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        data = resp.json()

        if overs_key in data:
            all_overs.extend(data[overs_key])

        last_response = data

        next_path = data.get("nextPaginationURL")
        if next_path:
            url = base_url + next_path
            time.sleep(delay)
        else:
            url = None

    final_data = last_response.copy() if last_response else {}
    final_data["overs"] = all_overs
    final_data.pop("nextPaginationURL", None)

    return final_data

get_all_overs(match_id=129585, innings_id=3, headers=DEFAULT_HEADERS, delay=0)

Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/3
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/3/1788016057773
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/3/1788011665914
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/3/1787938373551
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/3/1787934954104
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/3/1787932002357
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/2/1787927617364
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/2/1787923140011
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/2/1787917806901
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/2/1787915522196
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/1/1787911470676
Fetching: https://www.cricbuzz.com/api/mcenter/over-by-over/129585/1/1787848229163
Fetching: https://

{'paginatedData': [],
 'overs': [{'inningsId': 3,
   'overs': 47,
   'runs': 3,
   'score': 177,
   'wickets': 7,
   'ovrSummary': '1 1 1 0 0 ',
   'timestamp': 1788019635112,
   'batTeamName': 'ENG',
   'event': 'over-break',
   'batStrikerIds': [10385, 9531],
   'batStrikerNames': ['Ollie Robinson', 'Dan Lawrence'],
   'batStrikerRuns': 50,
   'batStrikerBalls': 66,
   'batNonStrikerIds': [],
   'batNonStrikerNames': [],
   'batNonStrikerRuns': 2,
   'batNonStrikerBalls': 2,
   'bowlIds': [35250],
   'bowlNames': ['Mohammad Ali'],
   'bowlOvers': 12.5,
   'bowlMaidens': 1,
   'bowlRuns': 54,
   'bowlWickets': 2},
  {'inningsId': 3,
   'overs': 46,
   'runs': 12,
   'score': 174,
   'wickets': 7,
   'ovrSummary': '0 4 2 0 0 6 ',
   'timestamp': 1788019393317,
   'batTeamName': 'ENG',
   'event': 'over-break',
   'batStrikerIds': [10385],
   'batStrikerNames': ['Dan Lawrence'],
   'batStrikerRuns': 49,
   'batStrikerBalls': 63,
   'batNonStrikerIds': [9531],
   'batNonStrikerNames': ['

In [17]:
def get_live_graph(match_id, innings_id, headers=None):
    response = requests.get(f'https://www.cricbuzz.com/api/mcenter/balls-map/{match_id}/{innings_id}', headers=DEFAULT_HEADERS)
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch live graph for match_id {match_id}, innings_id {innings_id}: {response.status_code}")

    return response.json()

get_live_graph(match_id=129585, innings_id=3, headers=DEFAULT_HEADERS)

{'balls': [{'timestamp': 1788019635112,
   'ballNbr': 281,
   'overNum': 46.5,
   'inningsId': 3,
   'event': 'NONE',
   'totalRuns': 0,
   'batsmanStrikerId': 10385,
   'bowlerStrikerId': 35250,
   'ballLabel': '•'},
  {'timestamp': 1788019586032,
   'ballNbr': 280,
   'overNum': 46.4,
   'inningsId': 3,
   'event': 'NONE',
   'totalRuns': 0,
   'batsmanStrikerId': 10385,
   'bowlerStrikerId': 35250,
   'ballLabel': '•'},
  {'timestamp': 1788019546870,
   'ballNbr': 279,
   'overNum': 46.3,
   'inningsId': 3,
   'event': 'NONE',
   'totalRuns': 1,
   'batsmanStrikerId': 9531,
   'bowlerStrikerId': 35250,
   'ballLabel': '1'},
  {'timestamp': 1788019496885,
   'ballNbr': 278,
   'overNum': 46.2,
   'inningsId': 3,
   'event': 'FIFTY',
   'totalRuns': 1,
   'batsmanStrikerId': 10385,
   'bowlerStrikerId': 35250,
   'ballLabel': '1'},
  {'timestamp': 1788019449002,
   'ballNbr': 277,
   'overNum': 46.1,
   'inningsId': 3,
   'event': 'NONE',
   'totalRuns': 1,
   'batsmanStrikerId': 9531

In [22]:
import re
import json
import requests


def _extract_flight_chunks(html: str) -> dict:
    """
    Next.js streams data via self.__next_f.push([1, "chunkId:jsonPayload"]).
    This pulls out every chunk and decodes the JS-string escaping.
    """
    pattern = re.compile(r'self\.__next_f\.push\(\[1,"(.*?)"\]\)', re.DOTALL)
    chunks = {}
    for raw in pattern.findall(html):
        try:
            # Wrapping in quotes lets json decode \", \\, \uXXXX etc. for us
            decoded = json.loads('"' + raw + '"')
        except json.JSONDecodeError:
            continue

        m = re.match(r'^([0-9a-fA-F]+):(.*)$', decoded, re.DOTALL)
        if not m:
            continue
        chunk_id, value = m.group(1), m.group(2).strip()
        chunks[chunk_id] = value
    return chunks


def _find_player_props(node):
    """Recursively search decoded JSON for the dict holding playerData."""
    if isinstance(node, dict):
        if "playerData" in node:
            return node
        for v in node.values():
            found = _find_player_props(v)
            if found:
                return found
    elif isinstance(node, list):
        for item in node:
            found = _find_player_props(item)
            if found:
                return found
    return None


def get_player_data(player_id, player_slug, headers=None):
    """
    Fetch a Cricbuzz player profile page and extract the embedded
    player data (bio, batting/bowling career stats, rankings, career
    timeline, recent form, news) from the Next.js flight payload.
    """
    if headers is None:
        headers = {
            'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,'
                      'image/avif,image/webp,image/apng,*/*;q=0.8',
            'accept-language': 'en-US,en;q=0.9',
            'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36',
        }

    url = f'https://www.cricbuzz.com/profiles/{player_id}/{player_slug}'
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()

    chunks = _extract_flight_chunks(resp.text)

    for chunk_id, value in chunks.items():
        try:
            parsed = json.loads(value)
        except (json.JSONDecodeError, ValueError):
            continue  # some chunks aren't JSON (text chunks, module refs, etc.)

        props = _find_player_props(parsed)
        if props:
            return props

    raise ValueError("Could not locate player data in the page's flight payload")


if __name__ == "__main__":
    data = get_player_data(8359, "babar-azam")
    print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "battingStats": {
    "headers": [
      "ROWHEADER",
      "Test",
      "ODI",
      "T20",
      "IPL"
    ],
    "values": [
      {
        "values": [
          "Matches",
          "65",
          "143",
          "144",
          "0"
        ]
      },
      {
        "values": [
          "Innings",
          "119",
          "140",
          "136",
          "0"
        ]
      },
      {
        "values": [
          "Runs",
          "4682",
          "6626",
          "4596",
          "0"
        ]
      },
      {
        "values": [
          "Balls",
          "8484",
          "7652",
          "3590",
          "0"
        ]
      },
      {
        "values": [
          "Highest",
          "196",
          "158",
          "122",
          "0"
        ]
      },
      {
        "values": [
          "Average",
          "43.35",
          "53.44",
          "38.95",
          "0"
        ]
      },
      {
        "values": [
          "SR",
          "55.19",


In [21]:
import requests

cookies = {
    'cbzads': 'PK|not_set|not_set|LAHORE',
    'cbgeo': 'PK',
    'WZRK_G': '3408ee4fbe1a48abbcc86debc902ad90',
    '_pubcid': '662bdb67-c851-419a-afaf-bab858d47dce',
    '_cc_id': 'c3da2803133e067940fabf5b93ff8e4d',
    'panoramaId_expiry': '1788109139180',
    '_sharedid': '6f8ad486-2207-4416-b5ba-a2ca66ce9858',
    '_gid': 'GA1.2.326675178.1788023080',
    '_ga_83LXEV4P47': 'GS2.1.s1788023100$o1$g1$t1788024782$j46$l0$h0',
    '__gads': 'ID=3d25eed9794cb754:T=1788022761:RT=1788027517:S=ALNI_MZQuXvJ0LY6udY9gsQghudTi1y6IQ',
    '__gpi': 'UID=0000152d8e49362b:T=1788022761:RT=1788027517:S=ALNI_MZn8KcphaatSBweypCDbmyZVQibDg',
    '__eoi': 'ID=923f2fccb36999d8:T=1788022761:RT=1788027517:S=AA-AfjZdHQ-C_IlJ3wakGCq17ate',
    '_sharedid_cst': 'zix7LPQsHA%3D%3D',
    'cto_bidid': '_v597l9XJTJGZm56dEg3UGJ3JTJCaWdhN0pHTzY3ZzlqSElab3MlMkJYM1JwSEJIU2s5dTI0b1djeWZGVDlRYUxwWUVqeEp5JTJCY0hnNDNYVDNVRHBadiUyQjVyUE5VMThBJTJGV2NMZlpJUUxxSk93MVJJU2ZpeUFzQSUzRA',
    'cto_bundle': 'MUOtRF9YUm9xaU00byUyQld2N2U1VERnbGtodXpFV0Zabkx2dSUyQmZIOVNvTzZrM0U3RTBOYWtjWFdZYjJVRTl0S3VUWXd3Z3hmMnolMkZUTVl3aHJPNkd3UG1xVVhaSk1jNjN5WW9LejAzU1U0NHl4JTJGdVRmcHM2bXdnN3NYaWhReXpIUlR3c0pYYUwxdkNHZDNIUCUyRjdJZjZHSmtqc2V3JTNEJTNE',
    '_ga': 'GA1.1.835928469.1788022735',
    'WZRK_S_R5K-RKZ-896Z': '%7B%22p%22%3A30%2C%22s%22%3A1788022735%2C%22t%22%3A1788027808%7D',
    'FCCDCF': '%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%222235958f-56c3-4cd9-8391-4f43e92d07c4%5C%22%2C%5B1788022734%2C788000000%5D%5D%22%5D%5D%5D',
    'FCNEC': '%5B%5B%22AKsRol8xgrt89SqjscwmWHHVlL_wqNgVvyw9yTXUdlNZe4AYIS7jkJogJ-2YGd7XkdW2UhcIxS6m4CtQKNQXFOafv_gXjj6EwMFTNWPUWDiPrJfaLFpYNLuJQIbcdyQ_FGuUJPTLtUlhSGXOvZyuwfUom9WcD1O_8Q%3D%3D%22%5D%5D',
    '_ga_4H06J8XXQH': 'GS2.1.s1788022735$o1$g1$t1788027811$j16$l0$h0',
}

headers = {
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'en-US,en;q=0.9',
    'cache-control': 'max-age=0',
    'priority': 'u=0, i',
    'referer': 'https://www.cricbuzz.com/cricket-match-squads/129585/eng-vs-pak-2nd-test-pakistan-tour-of-england-2026',
    'sec-ch-ua': '"Chromium";v="152", "Not?A_Brand";v="24", "Google Chrome";v="152"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36',
    # 'cookie': 'cbzads=PK|not_set|not_set|LAHORE; cbgeo=PK; WZRK_G=3408ee4fbe1a48abbcc86debc902ad90; _pubcid=662bdb67-c851-419a-afaf-bab858d47dce; _cc_id=c3da2803133e067940fabf5b93ff8e4d; panoramaId_expiry=1788109139180; _sharedid=6f8ad486-2207-4416-b5ba-a2ca66ce9858; _gid=GA1.2.326675178.1788023080; _ga_83LXEV4P47=GS2.1.s1788023100$o1$g1$t1788024782$j46$l0$h0; __gads=ID=3d25eed9794cb754:T=1788022761:RT=1788027517:S=ALNI_MZQuXvJ0LY6udY9gsQghudTi1y6IQ; __gpi=UID=0000152d8e49362b:T=1788022761:RT=1788027517:S=ALNI_MZn8KcphaatSBweypCDbmyZVQibDg; __eoi=ID=923f2fccb36999d8:T=1788022761:RT=1788027517:S=AA-AfjZdHQ-C_IlJ3wakGCq17ate; _sharedid_cst=zix7LPQsHA%3D%3D; cto_bidid=_v597l9XJTJGZm56dEg3UGJ3JTJCaWdhN0pHTzY3ZzlqSElab3MlMkJYM1JwSEJIU2s5dTI0b1djeWZGVDlRYUxwWUVqeEp5JTJCY0hnNDNYVDNVRHBadiUyQjVyUE5VMThBJTJGV2NMZlpJUUxxSk93MVJJU2ZpeUFzQSUzRA; cto_bundle=MUOtRF9YUm9xaU00byUyQld2N2U1VERnbGtodXpFV0Zabkx2dSUyQmZIOVNvTzZrM0U3RTBOYWtjWFdZYjJVRTl0S3VUWXd3Z3hmMnolMkZUTVl3aHJPNkd3UG1xVVhaSk1jNjN5WW9LejAzU1U0NHl4JTJGdVRmcHM2bXdnN3NYaWhReXpIUlR3c0pYYUwxdkNHZDNIUCUyRjdJZjZHSmtqc2V3JTNEJTNE; _ga=GA1.1.835928469.1788022735; WZRK_S_R5K-RKZ-896Z=%7B%22p%22%3A30%2C%22s%22%3A1788022735%2C%22t%22%3A1788027808%7D; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%222235958f-56c3-4cd9-8391-4f43e92d07c4%5C%22%2C%5B1788022734%2C788000000%5D%5D%22%5D%5D%5D; FCNEC=%5B%5B%22AKsRol8xgrt89SqjscwmWHHVlL_wqNgVvyw9yTXUdlNZe4AYIS7jkJogJ-2YGd7XkdW2UhcIxS6m4CtQKNQXFOafv_gXjj6EwMFTNWPUWDiPrJfaLFpYNLuJQIbcdyQ_FGuUJPTLtUlhSGXOvZyuwfUom9WcD1O_8Q%3D%3D%22%5D%5D; _ga_4H06J8XXQH=GS2.1.s1788022735$o1$g1$t1788027811$j16$l0$h0',
}

response = requests.get('https://www.cricbuzz.com/profiles/8359/babar-azam', cookies=cookies, headers=headers)

print(response.text)

<!DOCTYPE html><html lang="en" class="inter-app"><head><meta charSet="utf-8"/><link rel="preconnect" href="https://fonts.googleapis.com"/><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin="anonymous"/><link rel="dns-prefetch" href="https://static.cricbuzz.com/"/><link rel="preconnect" href="https://static.cricbuzz.com/"/><link rel="dns-prefetch" href="https://willow-static.cricbuzz.com/"/><link rel="preconnect" href="https://willow-static.cricbuzz.com/"/><link rel="stylesheet" href="/_next/static/css/0fbc1d072d12a5a0.css" data-precedence="next"/><link rel="preload" href="/_next/static/chunks/webpack-65fefaa13c9e7d05.js" as="script"/><link rel="preload" href="/_next/static/chunks/bce60fc1-80197ea5258fe292.js" as="script"/><link rel="preload" href="/_next/static/chunks/7698-e9795799c7804caf.js" as="script"/><link rel="preload" href="/_next/static/chunks/main-app-4b25eb3b7c4a6f84.js" as="script"/><link rel="preload" as="script" href="https://securepubads.g.doubleclick.ne